In [1]:
# %pip install -q python-terrier
# %pip install -q --upgrade git+https://github.com/terrierteam/pyterrier_t5.git

DEPRECATION: pytorch-lightning 1.5.10 has a non-standard dependency specifier torch>=1.7.*. pip 24.0 will enforce this behaviour change. A possible replacement is to upgrade to a newer version of pytorch-lightning or contact the author to suggest that they release a version with a conforming dependency specifiers. Discussion can be found at https://github.com/pypa/pip/issues/12063
Note: you may need to restart the kernel to use updated packages.
DEPRECATION: pytorch-lightning 1.5.10 has a non-standard dependency specifier torch>=1.7.*. pip 24.0 will enforce this behaviour change. A possible replacement is to upgrade to a newer version of pytorch-lightning or contact the author to suggest that they release a version with a conforming dependency specifiers. Discussion can be found at https://github.com/pypa/pip/issues/12063
Note: you may need to restart the kernel to use updated packages.


In [1]:
import pyterrier as pt
from pyterrier_dr import TctColBert, FlexIndex
from pyterrier_adaptive import GAR
from typing import Optional
import numpy as np
from collections import Counter
import pyterrier as pt
import pandas as pd
import ir_datasets
# Typing imports for type annotations
from typing import Any, Dict, List, Set, Tuple
import sys
import math
import warnings
import itertools
from collections import defaultdict
from pyterrier.model import add_ranks
import torch
from torch.nn import functional as F
from transformers import T5Config, T5Tokenizer, T5ForConditionalGeneration
from pyterrier.transformer import TransformerBase
import re
import scipy.sparse
import torch
from torch_geometric.utils import from_scipy_sparse_matrix
import scipy
from pyterrier_t5 import MonoT5ReRanker

from utils import *
from GNN import *

logger = ir_datasets.log.easy()
if not pt.started(): 
    pt.init()

/home/peppe/miniconda3/envs/GNRR/lib/python3.8/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
PyTerrier 0.10.0 has loaded Terrier 5.8 (built by craigm on 2023-11-01 18:05) and terrier-helper 0.0.8

No etc/terrier.properties, using terrier.default.properties for bootstrap configuration.


In [2]:
dataset = pt.get_dataset("irds:cord19/trec-covid")
index_loc = "./cord19-index"
# indexer = pt.IterDictIndexer(index_loc)
# indexref = indexer.index(dataset.get_corpus_iter(), fields=('title', 'abstract'))
indexref = pt.IndexRef.of("./cord19-index")

k = 8
flex_index = FlexIndex(index_path=f'./corpus_index/')

# Generate the corpus graph using the corpus_graph method of FlexIndex.
graph = flex_index.corpus_graph(k=k, batch_size=8192)

In [3]:
bm25 = pt.BatchRetrieve(indexref, wmodel="BM25", num_results=1000) 
monoT5 = MonoT5ReRanker(text_field='abstract')
TCTC = TctColBert('castorini/tct_colbert-msmarco')
mono_pipeline = pt.text.get_text(dataset, "abstract") >> monoT5


18:22:54.824 [main] WARN org.terrier.structures.FSADocumentIndex - This index has fields, but FSADocumentIndex is used (which stores fields lengths on disk); If using field-based models such as BM25F, change to index.document.class in the index  properties file to FSAFieldDocumentIndex or FSADocumentIndexInMemFields to support efficient retrieval. If you don't use (e.g.) BM25F, this warning can be ignored


/home/peppe/miniconda3/envs/GNRR/lib/python3.8/site-packages/transformers/models/t5/tokenization_t5.py:240: FutureWarning: This tokenizer was incorrectly instantiated with a model max length of 512 which will be corrected in Transformers v5.
For now, this behavior is kept to avoid breaking backwards compatibility when padding/encoding with `truncation is True`.
- Be aware that you SHOULD NOT rely on t5-base automatically truncating your input to 512 when padding/encoding.
- If you want to encode/pad to sequences longer than 512 you can either instantiate this tokenizer with `model_max_length` or pass `max_length` when encoding/padding.
- To avoid this warning, please instantiate this tokenizer with `model_max_length` set to your preferred value.
  warnings.warn(
You are using the default legacy behaviour of the <class 'transformers.models.t5.tokenization_t5.T5Tokenizer'>. This is expected, and simply means that the `legacy` (previous) behavior will be used so nothing changes for you. I

In [11]:
import torch
import pandas as pd
import copy
class TCT_scorer(TransformerBase):
    def __init__(self,
                 batch_size=16,
                 text_field='text',
                 config = None,
                 verbose=True):
        self.verbose = verbose
        self.batch_size = batch_size
        self.device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
        self.model_name = 'castorini/tct_colbert-msmarco'
        self.encoder = TctColBert(self.model_name, device = self.device)
        self.text_field = text_field
        n = 768
        input_features = n if config.aggr != 'concat' else n * 2

        if config.modality == 'local':
            mod = GNN_LG(input_features, config, modality = config.modality, conv_type = config.conv_type, device = self.device)
        elif config.modality == 'global':
            print("Work in progress...")
        elif config.modality == 'single':

            if config.conv_type != 'mlp':
                mod = GNN_NR(input_features, config, output_dim = 1, device=self.device)
            else:
                mod = MLP(input_features, config.hidden_dim, output_dim = 1, n_layers=config.n_layers, device=self.device, dropout_prob=config.dropout_prob)
        
        
        mod_copy = copy.deepcopy(mod)
        
        
        training_module = TrainingModule.load_from_checkpoint(checkpoint_path=config.model_path, model=mod_copy, lr=config.lr, wd=config.wd, aggr=config.aggr, model_family=config.conv_type, loss_type='listmle', ndcgk=[10, 20], recall=[10, 20], precision=[10, 20])
        t = torch.randn(1000, input_features)
        self.model = training_module.model

        initial_weights = {name: param.clone() for name, param in mod.named_parameters()}
        final_weights = {name: param.clone() for name, param in self.model.named_parameters()}

        for (name1, param1), (name2, param2) in zip(initial_weights.items(), final_weights.items()):
            if not torch.allclose(param1, param2):
                print(f"Weights of {name1}/{name2} have changed.")
            else:
                print(f"Weights of {name1}/{name2} have not changed.")

        out1 = self.model(t.clone())
        out2 = mod(t.clone())

        # print(out1)

        # print(out2)

        print(out1 - out2)

        self.config = config

    def __str__(self):
        return f"TCTscorer({self.model_name})"

    def transform(self, run, corpus_graph):
        
        # CHANGE
        topk_documents_df = run.drop_duplicates(subset='docno')
        queries, texts = topk_documents_df['query'], topk_documents_df[self.text_field]
  
        docs = texts
        # print(query)
        query_enc = self.encoder.encode_queries(queries.iloc[0:1])[0]
      
        doc_encs = self.encoder.encode_docs(docs)

        # print("before: ", len(run['docno'].unique()))
        
        
        
        # print("after: ", len(topk_documents_df['docno'].unique()))


        corpus_sb = self.generate_corpus_subgraph_induced_by_query(topk_documents_df = topk_documents_df, complete_corpus_graph = corpus_graph)

        # print("Corpus length: ", len(corpus_sb))

        # CHANGE
        docno_to_index = {docno: idx for idx, docno in enumerate(topk_documents_df['docno'].unique())}
        
        index_to_docno = {docno_to_index[k]: k for k in docno_to_index}
        

        # print("Doc idx length: ", len(docno_to_index))

        adj_matrix = self.build_adjacency_matrix(corpus_sb, docno_to_index)

        adj_matrix = adjacency_matrix_to_coo(adj_matrix).to(self.device)
        
        query_feat = (torch.from_numpy(query_enc).unsqueeze(0).unsqueeze(0)).to(self.device)
        x = (torch.from_numpy(doc_encs).unsqueeze(0)).to(self.device)

        A = adj_matrix.unsqueeze(0)


        if self.config.aggr == 'concat':
            rep_query = torch.repeat_interleave(query_feat, repeats=x.shape[1], dim=1)

            x = torch.cat((x, rep_query), dim = -1)

        elif self.config.aggr == 'sum':

            # print(query_feat.shape)
            rep_query = torch.repeat_interleave(query_feat, repeats=x.shape[1], dim=1)
            # print(rep_query.shape)
            x = x + rep_query
       
        elif self.config.aggr == 'hadamart':
        

            rep_query = torch.repeat_interleave(query_feat, repeats=x.shape[1], dim=1)
        
            # print(rep_query.shape)
            x = x * rep_query

        if self.config.conv_type == 'gcn' or self.config.conv_type == 'gat':
            out = self.model(x[0], A[0])
        else:
            out = self.model(x[0])
       

        out = out.squeeze()


        # Create a dictionary to store the data

        ordered_list = sorted(list(index_to_docno.keys()))
        topk_indices = torch.topk(out, k = out.shape[0]).indices
        data = {
            'qid': [run['qid'].iloc[0]]*len(ordered_list),
            'query': queries,
            'docno': [],
            'score': [], 
            'rank': []
        }

        for doc in ordered_list:

            data['docno'].append(index_to_docno[doc])
            data['score'].append(out[doc].item())
            data['rank'].append(topk_indices.tolist().index(doc) + 1)
            


        # Create the dataframe
        df = pd.DataFrame(data)

        # max_score = df['score'].max()
        # # max_rank = df[df['score'] == max_score]['rank'].values[0]
        # print("Maximum score:", max_score)
        # print("Rank of maximum score:", max_rank)
        
        
        return df
    
    def generate_corpus_subgraph_induced_by_query(
        self,                                           
        topk_documents_df: pd.DataFrame,
        complete_corpus_graph
    ) -> Dict[str, List[str]]:
        """
        Constructs and refines a corpus subgraph, focusing on relationships within a document subset.

        Conceptual Steps:
        1. Define Nodes: Identify and set the documents of interest as nodes in our subgraph. This step
        uses the 'documents_df' to extract document numbers, which will serve as nodes.

        2. Draw Edges: For each node, retrieve potential connections (edges) from the complete corpus graph.
        This involves fetching neighbors for each document from the comprehensive graph structure.

        3. Filter Edges: Refine the connections by ensuring each node (document) only connects to other nodes
        (documents) within our subset. This filtering process removes edges that lead outside the
        specified subset, maintaining the subgraph's integrity.

        4. Construct Subgraph: Populate the subgraph with nodes and their valid, filtered connections. This
        results in a dictionary where each key is a document number, and its value is a list of neighbor
        document numbers—all within the subset (i.e., valid neighbours).

        Args:
        - documents_df (pd.DataFrame): DataFrame containing documents of interest, identified by 'docno'.
        - graph_reference (NpTopKCorpusGraph): The complete corpus graph for neighbor retrieval.

        Returns:
        - Dict[str, List[str]]: Represents the corpus subgraph. Keys are document numbers ('docno'),
        and values are lists of neighbor document numbers, ensuring all are within the specified subset.
        """

        # Step 1: Define Nodes
        # Extract a set of document numbers to serve as valid nodes within our subgraph.
        valid_docnos = set(topk_documents_df['docno'])

        # Initialize the subgraph
        corpus_subgraph = {}
        found = 0
        for docno in valid_docnos:
            # Step 2: Draw Edges
            # Retrieve neighbors for the current document from the complete corpus graph.
            # CHANGE
            try:
                all_neighbors = complete_corpus_graph.neighbours(docno)
                found += 1
            except LookupError:
                # warnings.warn(f"Document {docno} not found in the corpus graph.")
                continue
            # Step 3: Filter Edges
            # Filter these neighbors to include only those also present in our subset (valid_docnos).
            valid_neighbors = [neighbor for neighbor in all_neighbors if neighbor in valid_docnos]

            # Step 4: Construct Subgraph
            # Update our subgraph to include the current document and its filtered neighbors.
            corpus_subgraph[docno] = valid_neighbors  # Populate subgraph
        
        return corpus_subgraph
    
    def build_adjacency_matrix(self, subgraph: Dict[str, list], docno_to_index: Dict[str, int]) -> np.ndarray:
        """
        Generates an adjacency matrix from a subgraph and a mapping of document numbers to indices.

        Parameters:
        subgraph (Dict[str, list]): A dictionary representing the subgraph with document numbers as keys.
        docno_to_index (Dict[str, int]): A dictionary mapping document numbers to their respective indices.

        Returns:
        np.ndarray: A symmetric adjacency matrix representing the graph.
        """
        # Error handling: Check if inputs are dictionaries
        if not isinstance(subgraph, dict) or not isinstance(docno_to_index, dict):
            raise ValueError("Both subgraph and docno_to_index must be dictionaries.")

        # Determine the size of the adjacency matrix
        # CHANGE
        matrix_size = len(docno_to_index)
        
        adjacency_matrix = np.zeros((matrix_size, matrix_size), dtype=int)

        # Iterate over each document and its neighbors in the subgraph
        for doc, neighbors in subgraph.items():
            if doc not in docno_to_index:
                raise KeyError(f"Document number {doc} not found in docno_to_index mapping.")
            doc_index = docno_to_index[doc]

            for neighbor in neighbors:
                if neighbor not in docno_to_index:
                    raise KeyError(f"Neighbor {neighbor} of document {doc} not found in docno_to_index mapping.")
                
                neighbor_index = docno_to_index[neighbor]

                # Mark the connection in the matrix, ensuring symmetry
                adjacency_matrix[doc_index, neighbor_index] = adjacency_matrix[neighbor_index, doc_index] = 1

        return adjacency_matrix
    
    
def adjacency_matrix_to_coo(adjacency_matrix: np.ndarray) -> torch.Tensor:
    """
    Converts an adjacency matrix to COO format using PyTorch Geometric.

    Parameters:
    adjacency_matrix (np.ndarray): The adjacency matrix to be converted.

    Returns:
    torch.Tensor: Edge index tensor in COO format.
    """
    # Convert the numpy adjacency matrix to a SciPy sparse matrix (COO format)
    scipy_sparse_matrix = scipy.sparse.coo_matrix(adjacency_matrix)

    # Convert the SciPy sparse matrix to PyTorch Geometric COO format
    edge_index, edge_weight = from_scipy_sparse_matrix(scipy_sparse_matrix)

    return edge_index

In [12]:
class GAR_mod(pt.Transformer):
    """
    A transformer that implements the Graph-based Adaptive Re-ranker algorithm from
    MacAvaney et al. "Adaptive Re-Ranking with a Corpus Graph" CIKM 2022.

    Required input columns: ['qid', 'query', 'docno', 'score', 'rank']
    Output columns: ['qid', 'query', 'docno', 'score', 'rank', 'iteration']
    where iteration defines the batch number which identified the document. Specifically
    even=initial retrieval   odd=corpus graph    -1=backfilled
    
    """
    def __init__(self,
        scorer: pt.Transformer,
        corpus_graph: 'CorpusGraph',
        num_results: int = 1000,
        batch_size: Optional[int] = None,
        backfill: bool = True,
        enabled: bool = True,
        verbose: bool = False):
        """
            GAR init method
            Args:
                scorer(pyterrier.Transformer): A transformer that scores query-document pairs. It will only be provided with ['qid, 'query', 'docno', 'score'].
                corpus_graph(pyterrier_adaptive.CorpusGraph): A graph of the corpus, enabling quick lookups of nearest neighbours
                num_results(int): The maximum number of documents to score (called "budget" and $c$ in the paper)
                batch_size(int): The number of documents to score at once (called $b$ in the paper). If not provided, will attempt to use the batch size from the scorer
                backfill(bool): If True, always include all documents from the initial stage, even if they were not re-scored
                enabled(bool): If False, perform re-ranking without using the corpus graph
                verbose(bool): If True, print progress information
        """
        self.scorer = scorer
        self.corpus_graph = corpus_graph
        self.num_results = num_results
        if batch_size is None:
            batch_size = scorer.batch_size if hasattr(scorer, 'batch_size') else 16
        self.batch_size = batch_size
        self.backfill = backfill
        self.enabled = enabled
        self.verbose = verbose

    def transform(self, df: pd.DataFrame) -> pd.DataFrame:
        """
        Applies Graph-based Adaptive Re-ranking to the provided dataframe. Essentially,
        Algorithm 1 from the paper.
        """
        result = {'qid': [], 'query': [], 'docno': [], 'rank': [], 'score': [], 'iteration': []}

        df = dict(iter(df.groupby(by=['qid'])))
        qids = df.keys()
        if self.verbose:
            qids = logger.pbar(qids, desc='adaptive re-ranking', unit='query')

        for qid in qids:
            query = df[qid]['query'].iloc[0]
            scores = {}
            res_map = [Counter(dict(zip(df[qid].docno, df[qid].score)))] # initial results
            if self.enabled:
                res_map.append(Counter()) # frontier
            frontier_data = {'minscore': float('inf')}
            iteration = 0
            while len(scores) < self.num_results and any(r for r in res_map):
                if len(res_map[iteration%len(res_map)]) == 0:
                    # if there's nothing available for the one we select, skip this iteration (i.e., move on to the next one)
                    iteration += 1
                    continue
                this_res = res_map[iteration%len(res_map)] # alternate between the initial ranking and frontier
                size = min(self.batch_size, self.num_results - len(scores)) # get either the batch size or remaining budget (whichever is smaller)
                
                # build batch of documents to score in this round
                batch = this_res.most_common(size)
                batch = pd.DataFrame(batch, columns=['docno', 'score'])
                batch['qid'] = qid
                batch['query'] = query

                # go score the batch of document with the re-ranker
                batch = self.scorer(batch)
                scores.update({k: (s, iteration) for k, s in zip(batch.docno, batch.score)})
                self._drop_docnos_from_counters(batch.docno, res_map)
                if len(scores) < self.num_results and self.enabled:
                    self._update_frontier(batch, res_map[1], frontier_data, scores)
                iteration += 1

            # Add scored items to results
            result['qid'].append(np.full(len(scores), qid))
            result['query'].append(np.full(len(scores), query))
            result['rank'].append(np.arange(len(scores)))
            for did, (score, i) in Counter(scores).most_common():
                result['docno'].append(did)
                result['score'].append(score)
                result['iteration'].append(i)

            # Backfill unscored items
            if self.backfill and len(scores) < self.num_results:
                last_score = result['score'][-1] if result['score'] else 0.
                count = min(self.num_results - len(scores), len(res_map[0]))
                result['qid'].append(np.full(count, qid))
                result['query'].append(np.full(count, query))
                result['rank'].append(np.arange(len(scores), len(scores) + count))
                for i, (did, score) in enumerate(res_map[0].most_common()):
                    if i >= count:
                        break
                    result['docno'].append(did)
                    result['score'].append(last_score - 1 - i)
                    result['iteration'].append(-1)

        display(pd.DataFrame({
            'qid': np.concatenate(result['qid']),
            'query': np.concatenate(result['query']),
            'docno': result['docno'],
            'rank': np.concatenate(result['rank']),
            'score': result['score'],
            'iteration': result['iteration'],
        }))
        print(result['rank'])
        return pd.DataFrame({
            'qid': np.concatenate(result['qid']),
            'query': np.concatenate(result['query']),
            'docno': result['docno'],
            'rank': np.concatenate(result['rank']),
            'score': result['score'],
        })
    def _update_frontier(self, scored_batch, frontier, frontier_data, scored_dids):
        remaining_budget = self.num_results - len(scored_dids)
        for score, did in sorted(zip(scored_batch.score, scored_batch.docno), reverse=True):
            if len(frontier) < remaining_budget or score >= frontier_data['minscore']:
                hit = False
                try:
                    for target_did in self.corpus_graph.neighbours(did):
                        if target_did not in scored_dids:
                            if target_did not in frontier or score > frontier[target_did]:
                                frontier[target_did] = score
                                hit = True
                    if hit and score < frontier_data['minscore']:
                        frontier_data['minscore'] = score
                except LookupError:
                    continue

    def _drop_docnos_from_counters(self, docnos, counters):
        for docno in docnos:
            for c in counters:
                del c[docno]

class GNRR(pt.Transformer):
    """
    A transformer that implements the Graph-based Adaptive Re-ranker algorithm from
    MacAvaney et al. "Adaptive Re-Ranking with a Corpus Graph" CIKM 2022.

    Required input columns: ['qid', 'query', 'docno', 'score', 'rank']
    Output columns: ['qid', 'query', 'docno', 'score', 'rank', 'iteration']
    where iteration defines the batch number which identified the document. Specifically
    even=initial retrieval   odd=corpus graph    -1=backfilled
    
    """
    def __init__(self,
        scorer: pt.Transformer,
        corpus_graph: 'CorpusGraph',
        text_field = 'abstract',
        num_results: int = 1000,
        batch_size: Optional[int] = None,
        backfill: bool = True,
        enabled: bool = True,
        verbose: bool = False):
        """
            GAR init method
            Args:
                scorer(pyterrier.Transformer): A transformer that scores query-document pairs. It will only be provided with ['qid, 'query', 'docno', 'score'].
                corpus_graph(pyterrier_adaptive.CorpusGraph): A graph of the corpus, enabling quick lookups of nearest neighbours
                num_results(int): The maximum number of documents to score (called "budget" and $c$ in the paper)
                batch_size(int): The number of documents to score at once (called $b$ in the paper). If not provided, will attempt to use the batch size from the scorer
                backfill(bool): If True, always include all documents from the initial stage, even if they were not re-scored
                enabled(bool): If False, perform re-ranking without using the corpus graph
                verbose(bool): If True, print progress information
        """
        self.scorer = scorer
        self.corpus_graph = corpus_graph
        self.text_field = text_field
        self.num_results = num_results
        if batch_size is None:
            batch_size = scorer.batch_size if hasattr(scorer, 'batch_size') else 16
        self.batch_size = batch_size
        self.backfill = backfill
        self.enabled = enabled
        self.verbose = verbose
        

    def transform(self, df: pd.DataFrame) -> pd.DataFrame:
        """
        Applies Graph-based Adaptive Re-ranking to the provided dataframe. Essentially,
        Algorithm 1 from the paper.
        """
        result = {'qid': [], 'query': [], 'docno': [], 'rank': [], 'score': []}

        result = pd.DataFrame(result)

        df = dict(iter(df.groupby(by=['qid'])))
        qids = df.keys()

        if self.verbose:
            qids = logger.pbar(qids, desc='adaptive re-ranking', unit='query')
        
        for qid in qids:
            print("Now we study the query: ", qid)
            
            batch = df[qid].loc[:, ['qid', 'query', 'docno', 'score']]
                # go score the batch of document with the re-ranker
            add_texts = pt.text.get_text(dataset, self.text_field)
            batch = add_texts(batch)

            # print(batch)
            inter_result = self.scorer.transform(batch, self.corpus_graph)
            display(inter_result)

            result = pd.concat([result, inter_result], axis=0, ignore_index=True)
       

            result['rank'] = result['rank'].astype(int)

        return result

In [13]:
class Config:
    def __init__(self, modality, conv_type, hidden_dim, dropout_prob, n_layers, aggr, neighbors, model_path, heads, lr, wd):
        self.modality = modality
        self.conv_type = conv_type
        self.hidden_dim = hidden_dim
        self.dropout_prob = dropout_prob
        self.n_layers = n_layers
        self.aggr = aggr
        self.neighbors = neighbors
        self.heads = heads
        self.lr = lr
        self.wd = wd
        self.model_path = model_path



config_local_gcn = Config(modality='local', conv_type='gcn', hidden_dim=256, dropout_prob=0,
                n_layers=2, aggr='concat', neighbors=True, model_path='local_gcn_concat.ckpt', heads=1, lr=0.001, wd=0.01)


config_single_mlp = Config(modality='single', conv_type='mlp', hidden_dim=256, dropout_prob=0,
                n_layers=2, aggr='concat', neighbors=True, model_path='single_mlp_concat.ckpt', heads=1, lr=0.001, wd=0.01)

In [14]:
# tct_scorer_local_gcn = TCT_scorer(config = config_local_gcn, text_field='abstract')
tct_scorer_single_mlp = TCT_scorer(config = config_single_mlp, text_field='abstract')


Weights of mlp.0.weight/mlp.0.weight have changed.
Weights of mlp.0.bias/mlp.0.bias have changed.
Weights of mlp.2.weight/mlp.2.weight have changed.
Weights of mlp.2.bias/mlp.2.bias have changed.
Weights of mlp.4.weight/mlp.4.weight have changed.
Weights of mlp.4.bias/mlp.4.bias have changed.
Weights of mlp.6.weight/mlp.6.weight have changed.
Weights of mlp.6.bias/mlp.6.bias have changed.
tensor([[ 2.5850e+00],
        [-2.5170e-01],
        [-2.0466e+00],
        [ 8.3098e-03],
        [ 8.7516e-02],
        [-6.3040e-01],
        [ 8.1332e-01],
        [ 8.3098e-03],
        [ 8.3098e-03],
        [ 7.6728e-02],
        [ 8.3098e-03],
        [ 8.3098e-03],
        [ 2.6317e+00],
        [ 5.1098e-01],
        [ 1.3039e+00],
        [ 8.3098e-03],
        [ 3.3524e+00],
        [-3.1236e-01],
        [-2.4750e+00],
        [ 8.1861e-01],
        [ 8.3098e-03],
        [ 8.3098e-03],
        [ 8.3098e-03],
        [ 8.3098e-03],
        [ 1.3535e+00],
        [-3.5124e+00],
        [ 

In [15]:
import json

# Load the test indices from the file
with open('test_indices.json', 'r') as f:
    test_indices = [14]#json.load(f)

# Filter the dataframe based on the 'qid' that belong to the test indices

filtered_get_topics = dataset.get_topics("description")[dataset.get_topics("description")['qid'].isin(list(map(str, test_indices)))]
filtered_get_qrels = dataset.get_qrels()[dataset.get_qrels()['qid'].isin(list(map(str, test_indices)))]

# dataset.get_topics("description")

# dataset.get_qrels()


In [16]:
from pyterrier.measures import * 

pt.Experiment(
  [ 
    #bm25 >> GAR_mod(pt.text.get_text(dataset, "abstract") >> monoT5, graph),
    bm25 >> GNRR(tct_scorer_single_mlp, graph, text_field='abstract'),  
    #bm25 >> GNRR(tct_scorer_local_gcn, graph, text_field='abstract'),
    #bm25 >> pt.text.get_text(dataset, "abstract") >> TCTC,
    #bm25 >> pt.text.get_text(dataset, "abstract") >> monoT5
  ],
  filtered_get_topics,
  filtered_get_qrels,
  names=["TCTColbert + MLP"],#, "TCTColbert+MLP+GCN"],#, "TCTColbert"],#, "MonoT5"],#, "MonoT5"],
  eval_metrics=[nDCG@10, nDCG@20, R@10, R@20, P@10, P@20, R(rel = 2)@10, R(rel = 2)@20, P(rel = 2)@10, P(rel = 2)@20, R(rel=2)@1000]
)

Now we study the query:  14


: 